# Make Baseline Predictions

Predict consumption using the previous hour and the same hour yesterday. Use the three-home example.

Open this notebook in `notebooks/M4/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [1]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction')

## Choose the existing dataset

In [2]:
from pathlib import Path
import pandas as pd
raw_path = PROJECT_ROOT / "data/raw/half_hour_readings_2013.csv"
raw_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/data/raw/half_hour_readings_2013.csv')

## Use M4's prototype

In [3]:
homes = ["MAC000038", "MAC000045", "MAC001617"]
start = pd.Timestamp("2013-01-01", tz="UTC")
end = pd.Timestamp("2013-01-08", tz="UTC")
homes, start, end

(['MAC000038', 'MAC000045', 'MAC001617'],
 Timestamp('2013-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2013-01-08 00:00:00+0000', tz='UTC'))

## Read only the three homes into memory

In [4]:
pieces = []
for chunk in pd.read_csv(raw_path, chunksize=200000, na_values=["Null", "null"]):
    pieces.append(chunk[chunk["LCLid"].isin(homes)])
data = pd.concat(pieces, ignore_index=True)
data.columns

Index(['LCLid', 'tstp', 'energy(kWh/hh)'], dtype='object')

## Parse the fields

In [5]:
data = data.rename(columns={"energy(kWh/hh)": "kwh"})
data["tstp"] = pd.to_datetime(data["tstp"], utc=True)
data["kwh"] = pd.to_numeric(data["kwh"])
data.dtypes

LCLid                 object
tstp     datetime64[ns, UTC]
kwh                  float64
dtype: object

## Keep exactly the first week's intervals

In [6]:
data = data[(data["tstp"] > start) & (data["tstp"] <= end)].copy()
data.head()

,LCLid,tstp,kwh
1,MAC000045,2013-01-01 00:30:00+00:00,0.029
2,MAC000045,2013-01-01 01:00:00+00:00,0.810
3,MAC000045,2013-01-01 01:30:00+00:00,0.677
4,MAC000045,2013-01-01 02:00:00+00:00,0.112
5,MAC000045,2013-01-01 02:30:00+00:00,0.108


## See the sample size per home

In [7]:
data.groupby("LCLid").size()

LCLid
MAC000038    336
MAC000045    336
MAC001617    326
dtype: int64

## Derive the hourly start

In [8]:
data["hour"] = (data["tstp"] - pd.Timedelta(minutes=30)).dt.floor("h")
data[["LCLid", "tstp", "hour", "kwh"]].head()

,LCLid,tstp,hour,kwh
1,MAC000045,2013-01-01 00:30:00+00:00,2013-01-01 00:00:00+00:00,0.029
2,MAC000045,2013-01-01 01:00:00+00:00,2013-01-01 00:00:00+00:00,0.810
3,MAC000045,2013-01-01 01:30:00+00:00,2013-01-01 01:00:00+00:00,0.677
4,MAC000045,2013-01-01 02:00:00+00:00,2013-01-01 01:00:00+00:00,0.112
5,MAC000045,2013-01-01 02:30:00+00:00,2013-01-01 02:00:00+00:00,0.108


## Add the two half-hours

In [9]:
grouped = data.groupby(["LCLid", "hour"])["kwh"].agg(["sum", "count"]).reset_index()
grouped.loc[grouped["count"] != 2, "sum"] = float("nan")
hourly_values = grouped.rename(columns={"sum": "kwh"})[["LCLid", "hour", "kwh"]]
grouped.head()

,LCLid,hour,sum,count
0,MAC000038,2013-01-01 00:00:00+00:00,1.188,2
1,MAC000038,2013-01-01 01:00:00+00:00,0.503,2
2,MAC000038,2013-01-01 02:00:00+00:00,0.164,2
3,MAC000038,2013-01-01 03:00:00+00:00,0.143,2
4,MAC000038,2013-01-01 04:00:00+00:00,0.128,2


## Make every expected hour visible

In [10]:
hours = pd.date_range(start, end, freq="h", inclusive="left")
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "hour"]).to_frame(index=False)
len(grid)

504

## Keep entirely absent hours as unknown

In [11]:
hourly = grid.merge(hourly_values, on=["LCLid", "hour"], how="left")
hourly.head()

,LCLid,hour,kwh
0,MAC000038,2013-01-01 00:00:00+00:00,1.188
1,MAC000038,2013-01-01 01:00:00+00:00,0.503
2,MAC000038,2013-01-01 02:00:00+00:00,0.164
3,MAC000038,2013-01-01 03:00:00+00:00,0.143
4,MAC000038,2013-01-01 04:00:00+00:00,0.128


## Count missing hours

In [12]:
hourly["kwh"].isna().groupby(hourly["LCLid"]).sum()

LCLid
MAC000038    0
MAC000045    0
MAC001617    6
Name: kwh, dtype: int64

## Prepare the actual values

In [13]:
lookup = hourly.set_index(["LCLid", "hour"])["kwh"]
predictions = hourly.rename(columns={"hour": "target_hour", "kwh": "actual_kwh"}).copy()
predictions.head()

,LCLid,target_hour,actual_kwh
0,MAC000038,2013-01-01 00:00:00+00:00,1.188
1,MAC000038,2013-01-01 01:00:00+00:00,0.503
2,MAC000038,2013-01-01 02:00:00+00:00,0.164
3,MAC000038,2013-01-01 03:00:00+00:00,0.143
4,MAC000038,2013-01-01 04:00:00+00:00,0.128


## Predict using the previous hour

In [14]:
previous_time = predictions["target_hour"] - pd.Timedelta(hours=1)
previous_keys = pd.MultiIndex.from_arrays([predictions["LCLid"], previous_time])
predictions["pred_prev_hour"] = lookup.reindex(previous_keys).to_numpy()
predictions.head()

,LCLid,target_hour,actual_kwh,pred_prev_hour
0,MAC000038,2013-01-01 00:00:00+00:00,1.188,NaN
1,MAC000038,2013-01-01 01:00:00+00:00,0.503,1.188
2,MAC000038,2013-01-01 02:00:00+00:00,0.164,0.503
3,MAC000038,2013-01-01 03:00:00+00:00,0.143,0.164
4,MAC000038,2013-01-01 04:00:00+00:00,0.128,0.143


## Predict using the same hour yesterday

In [15]:
yesterday_time = predictions["target_hour"] - pd.Timedelta(hours=24)
yesterday_keys = pd.MultiIndex.from_arrays([predictions["LCLid"], yesterday_time])
predictions["pred_prev_day"] = lookup.reindex(yesterday_keys).to_numpy()
predictions.iloc[22:28]

,LCLid,target_hour,actual_kwh,pred_prev_hour,pred_prev_day
22,MAC000038,2013-01-01 22:00:00+00:00,0.740,0.947,NaN
23,MAC000038,2013-01-01 23:00:00+00:00,0.645,0.740,NaN
24,MAC000038,2013-01-02 00:00:00+00:00,0.227,0.645,1.188
25,MAC000038,2013-01-02 01:00:00+00:00,0.111,0.227,0.503
26,MAC000038,2013-01-02 02:00:00+00:00,0.136,0.111,0.164
27,MAC000038,2013-01-02 03:00:00+00:00,0.131,0.136,0.143


## Check one row directly from raw data

In [16]:
home = "MAC000038"
target = pd.Timestamp("2013-01-02 01:00", tz="UTC")
hand = pd.DataFrame({"field": ["actual_kwh", "pred_prev_hour", "pred_prev_day"], "offset": [0, 1, 24]})
hand["hour"] = target - pd.to_timedelta(hand["offset"], unit="h")
hand

,field,offset,hour
0,actual_kwh,0,2013-01-02 01:00:00+00:00
1,pred_prev_hour,1,2013-01-02 00:00:00+00:00
2,pred_prev_day,24,2013-01-01 01:00:00+00:00


## Look up the two ends for each hour

In [17]:
raw_home = data[data["LCLid"] == home].set_index("tstp")["kwh"]
hand["first_end"] = hand["hour"] + pd.Timedelta(minutes=30)
hand["second_end"] = hand["hour"] + pd.Timedelta(hours=1)
hand["first_kwh"] = raw_home.reindex(hand["first_end"]).to_numpy()
hand["second_kwh"] = raw_home.reindex(hand["second_end"]).to_numpy()
hand["from_raw"] = hand["first_kwh"] + hand["second_kwh"]
hand

,field,offset,hour,first_end,second_end,first_kwh,second_kwh,from_raw
0,actual_kwh,0,2013-01-02 01:00:00+00:00,2013-01-02 01:30:00+00:00,2013-01-02 02:00:00+00:00,0.056,0.055,0.111
1,pred_prev_hour,1,2013-01-02 00:00:00+00:00,2013-01-02 00:30:00+00:00,2013-01-02 01:00:00+00:00,0.167,0.060,0.227
2,pred_prev_day,24,2013-01-01 01:00:00+00:00,2013-01-01 01:30:00+00:00,2013-01-01 02:00:00+00:00,0.318,0.185,0.503


## Compare the hand calculation

In [18]:
selected = predictions[(predictions["LCLid"] == home) & (predictions["target_hour"] == target)].iloc[0]
hand["from_predictions"] = selected[hand["field"]].to_numpy()
hand["matches"] = (hand["from_raw"] - hand["from_predictions"]).abs() < 1e-12
hand

,field,offset,hour,first_end,second_end,first_kwh,second_kwh,from_raw,from_predictions,matches
0,actual_kwh,0,2013-01-02 01:00:00+00:00,2013-01-02 01:30:00+00:00,2013-01-02 02:00:00+00:00,0.056,0.055,0.111,0.111,True
1,pred_prev_hour,1,2013-01-02 00:00:00+00:00,2013-01-02 00:30:00+00:00,2013-01-02 01:00:00+00:00,0.167,0.060,0.227,0.227,True
2,pred_prev_day,24,2013-01-01 01:00:00+00:00,2013-01-01 01:30:00+00:00,2013-01-01 02:00:00+00:00,0.318,0.185,0.503,0.503,True


## Inspect a gap

In [19]:
gap_start = pd.Timestamp("2013-01-06 19:00", tz="UTC")
gap_rows = predictions[(predictions["LCLid"] == "MAC001617") & (predictions["target_hour"] >= gap_start)]
gap_rows.head(9)

,LCLid,target_hour,actual_kwh,pred_prev_hour,pred_prev_day
475,MAC001617,2013-01-06 19:00:00+00:00,NaN,0.075,0.793
476,MAC001617,2013-01-06 20:00:00+00:00,NaN,NaN,0.669
477,MAC001617,2013-01-06 21:00:00+00:00,NaN,NaN,0.410
478,MAC001617,2013-01-06 22:00:00+00:00,NaN,NaN,0.335
479,MAC001617,2013-01-06 23:00:00+00:00,NaN,NaN,0.276
480,MAC001617,2013-01-07 00:00:00+00:00,0.293,NaN,0.239
481,MAC001617,2013-01-07 01:00:00+00:00,0.192,0.293,0.405
482,MAC001617,2013-01-07 02:00:00+00:00,0.054,0.192,0.203
483,MAC001617,2013-01-07 03:00:00+00:00,0.067,0.054,0.135


## Set up M4's invented baseline checks

In [20]:
test_hours = pd.date_range(start, periods=27, freq="h")
test = pd.DataFrame({"LCLid": "TEST", "hour": test_hours, "kwh": [i / 10 for i in range(27)]})
test.loc[25, "kwh"] = float("nan")
test = test.drop(index=3)
test.head()

,LCLid,hour,kwh
0,TEST,2013-01-01 00:00:00+00:00,0.0
1,TEST,2013-01-01 01:00:00+00:00,0.1
2,TEST,2013-01-01 02:00:00+00:00,0.2
4,TEST,2013-01-01 04:00:00+00:00,0.4
5,TEST,2013-01-01 05:00:00+00:00,0.5


## Calculate the invented previous-hour predictions

In [21]:
test_lookup = test.set_index(["LCLid", "hour"])["kwh"]
test_predictions = test.rename(columns={"hour": "target_hour", "kwh": "actual_kwh"}).copy()
test_previous = test_predictions["target_hour"] - pd.Timedelta(hours=1)
test_keys = pd.MultiIndex.from_arrays([test_predictions["LCLid"], test_previous])
test_predictions["pred_prev_hour"] = test_lookup.reindex(test_keys).to_numpy()
test_predictions.head()

,LCLid,target_hour,actual_kwh,pred_prev_hour
0,TEST,2013-01-01 00:00:00+00:00,0.0,NaN
1,TEST,2013-01-01 01:00:00+00:00,0.1,0.0
2,TEST,2013-01-01 02:00:00+00:00,0.2,0.1
4,TEST,2013-01-01 04:00:00+00:00,0.4,NaN
5,TEST,2013-01-01 05:00:00+00:00,0.5,0.4


## Calculate the invented previous-day predictions

In [22]:
test_yesterday = test_predictions["target_hour"] - pd.Timedelta(hours=24)
test_keys = pd.MultiIndex.from_arrays([test_predictions["LCLid"], test_yesterday])
test_predictions["pred_prev_day"] = test_lookup.reindex(test_keys).to_numpy()
test_results = test_predictions.set_index("target_hour")
test_results.tail(3)

,LCLid,actual_kwh,pred_prev_hour,pred_prev_day
target_hour,,,,
2013-01-02 00:00:00+00:00,TEST,2.4,2.3,0.0
2013-01-02 01:00:00+00:00,TEST,NaN,2.4,0.1
2013-01-02 02:00:00+00:00,TEST,2.6,NaN,0.2


## Check a normal hour and an absent hour

In [23]:
test_results.loc[test_hours[[2, 4]], ["actual_kwh", "pred_prev_hour"]]

,actual_kwh,pred_prev_hour
2013-01-01 02:00:00+00:00,0.2,0.1
2013-01-01 04:00:00+00:00,0.4,NaN


## Check the first hour and missing inputs

In [24]:
test_results.loc[test_hours[[0, 24, 26]], ["pred_prev_hour", "pred_prev_day"]]

,pred_prev_hour,pred_prev_day
2013-01-01 00:00:00+00:00,NaN,NaN
2013-01-02 00:00:00+00:00,2.3,0.0
2013-01-02 02:00:00+00:00,NaN,0.2


## Add the output labels

In [25]:
predictions.insert(0, "task_id", "M4-W07-T01")
predictions["label"] = "PROTOTYPE"
predictions.head()

,task_id,LCLid,target_hour,actual_kwh,pred_prev_hour,pred_prev_day,label
0,M4-W07-T01,MAC000038,2013-01-01 00:00:00+00:00,1.188,NaN,NaN,PROTOTYPE
1,M4-W07-T01,MAC000038,2013-01-01 01:00:00+00:00,0.503,1.188,NaN,PROTOTYPE
2,M4-W07-T01,MAC000038,2013-01-01 02:00:00+00:00,0.164,0.503,NaN,PROTOTYPE
3,M4-W07-T01,MAC000038,2013-01-01 03:00:00+00:00,0.143,0.164,NaN,PROTOTYPE
4,M4-W07-T01,MAC000038,2013-01-01 04:00:00+00:00,0.128,0.143,NaN,PROTOTYPE


## Choose the output folder

In [26]:
output_dir = PROJECT_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results')

## Save the predictions

In [27]:
prediction_path = output_dir / "baseline_predictions_sample.csv"
predictions.to_csv(prediction_path, index=False)
prediction_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/baseline_predictions_sample.csv')

## Save the hand-check evidence

In [28]:
hand_path = output_dir / "baseline_check_sample.csv"
hand.to_csv(hand_path, index=False)
hand_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/baseline_check_sample.csv')

# Week 2: Baselines on the shared validation rows

Use M3's shared input file instead of the three-home example. Only VALIDATION rows are used here.

## Read M3's shared inputs

In [29]:
inputs = pd.read_csv(PROJECT_ROOT / "data/processed/prediction_inputs.csv")
inputs["target_hour"] = pd.to_datetime(inputs["target_hour"], utc=True)
inputs.shape, inputs["period"].value_counts()

((2084732, 6),
 period
 TRAIN         1392091
 VALIDATION     349127
 TEST           343514
 Name: count, dtype: int64)

## Keep VALIDATION rows only

In [30]:
validation = inputs[inputs["period"] == "VALIDATION"].copy()
len(validation), validation["LCLid"].nunique(), validation["target_hour"].min(), validation["target_hour"].max()

(349127,
 240,
 Timestamp('2013-09-01 00:00:00+0000', tz='UTC'),
 Timestamp('2013-10-31 23:00:00+0000', tz='UTC'))

## Check the dates follow the agreed period

In [31]:
import json
rules = json.load(open(PROJECT_ROOT / "data/split_rules.json"))
val_start, val_end = [pd.Timestamp(day, tz="UTC") for day in rules["periods"]["VALIDATION"]]
((validation["target_hour"] >= val_start) & (validation["target_hour"] < val_end)).all()

True

## Check no reserved home is used

In [32]:
split = pd.read_csv(PROJECT_ROOT / "data/household_split.csv")
reserved = split.loc[split["Cohort"] == "RESERVED", "LCLid"]
validation["LCLid"].isin(reserved).sum()

0

## Copy the two inputs as the baseline predictions

In [33]:
val_predictions = validation.rename(columns={"target_kwh": "actual_kwh"})
val_predictions["pred_prev_hour"] = val_predictions["lag_1h_kwh"]
val_predictions["pred_prev_day"] = val_predictions["lag_24h_kwh"]
val_predictions = val_predictions[["LCLid", "target_hour", "period", "actual_kwh",
    "pred_prev_hour", "pred_prev_day"]]
val_predictions.head()

,LCLid,target_hour,period,actual_kwh,pred_prev_hour,pred_prev_day
5799,MAC000038,2013-09-01 00:00:00+00:00,VALIDATION,0.159,0.202,0.280
5800,MAC000038,2013-09-01 01:00:00+00:00,VALIDATION,0.124,0.159,0.243
5801,MAC000038,2013-09-01 02:00:00+00:00,VALIDATION,0.280,0.124,0.224
5802,MAC000038,2013-09-01 03:00:00+00:00,VALIDATION,0.170,0.280,0.361
5803,MAC000038,2013-09-01 04:00:00+00:00,VALIDATION,0.151,0.170,0.248


## Trace one row back to the inputs

In [34]:
row = val_predictions.iloc[0]
source = validation.iloc[0]
row["pred_prev_hour"] == source["lag_1h_kwh"], row["pred_prev_day"] == source["lag_24h_kwh"]

(True, True)

## Check each home/hour appears once

In [35]:
val_predictions.duplicated(["LCLid", "target_hour"]).sum()

0

## Add the output label

In [36]:
val_predictions.insert(0, "task_id", "M4-W2")
val_predictions.head()

,task_id,LCLid,target_hour,period,actual_kwh,pred_prev_hour,pred_prev_day
5799,M4-W2,MAC000038,2013-09-01 00:00:00+00:00,VALIDATION,0.159,0.202,0.280
5800,M4-W2,MAC000038,2013-09-01 01:00:00+00:00,VALIDATION,0.124,0.159,0.243
5801,M4-W2,MAC000038,2013-09-01 02:00:00+00:00,VALIDATION,0.280,0.124,0.224
5802,M4-W2,MAC000038,2013-09-01 03:00:00+00:00,VALIDATION,0.170,0.280,0.361
5803,M4-W2,MAC000038,2013-09-01 04:00:00+00:00,VALIDATION,0.151,0.170,0.248


## Save the validation predictions

In [37]:
val_prediction_path = output_dir / "validation_predictions.csv"
val_predictions.to_csv(val_prediction_path, index=False)
val_prediction_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/validation_predictions.csv')